# Capstone · A Research Agent from Scratch

Bring Phase 3 together into one agent that:

1. **uses tools** — a calculator *and* a retrieval `search` tool (agentic RAG over the Phase 2 corpus),
2. runs the **agent loop** (think → act → observe → answer),
3. **reflects** on its own answer and revises it, and
4. reports a **live USD cost meter** for every model call.

Everything is built on `study_buddy` (providers + cost), `rag_lab` (retrieval), and `agents_lab` (the agent machinery).

> Runs on Ollama end-to-end. Swap `PROVIDER` to a cloud model to see real USD costs.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'agents_lab').exists())
sys.path.insert(0, str(root))

PROVIDER = 'ollama'   # try 'openai' / 'anthropic' with a key set to see real cost

**Step 1 — Knowledge base.** Index the Phase 2 multi-hop corpus with local embeddings.

In [ ]:
from rag_lab import load_markdown, NumpyStore

corpus_dir = next(s / 'multihop' for s in root.parent.iterdir() if (s / 'multihop').is_dir())
store = NumpyStore(provider='ollama')
store.add(load_markdown(directory=corpus_dir))
print('chunks indexed:', len(store))

**Step 2 — Tools.** Combine the built-in calculator with a retrieval tool built from our store. One shared `CostTracker` meters every call.

In [ ]:
from agents_lab import Agent, ToolRegistry, default_tools, make_rag_tool, Tracer
from study_buddy import CostTracker

tools = ToolRegistry(list(default_tools()[n] for n in default_tools().names))
tools.add(make_rag_tool(store, k=4))

tracker = CostTracker()
SYSTEM = (
    'You are a research agent. Use `search` to look up facts in the knowledge base '
    'and `calculator` for arithmetic. Answer using ONLY retrieved facts and cite '
    '[source] tags. If a needed fact is missing, say you don\'t know.'
)
agent = Agent(provider=PROVIDER, tools=tools, system=SYSTEM, max_steps=6, tracker=tracker)
print('tools available:', tools.names)

**Step 3 — Run the loop** on a multi-hop question and watch the trajectory.

In [ ]:
question = "What year did the company that acquired the startup founded by Sam's college roommate go public?"
tracer = Tracer()
result = agent.run(question, tracer=tracer)
print(tracer.pretty())
print('\nDRAFT ANSWER:\n', result.answer)

**Step 4 — Reflect and revise.** Let the agent critique its own answer, then improve it if needed.

In [ ]:
from agents_lab import reflect
from study_buddy import get_provider

critique = reflect(question, result.answer, provider=PROVIDER, tracker=tracker)
print('CRITIQUE:\n', critique)

final = result.answer
if not critique.strip().upper().startswith('OK'):
    revise = get_provider(PROVIDER).chat(
        [
            {'role': 'system', 'content': 'Revise the answer to address the critique. Keep [source] citations.'},
            {'role': 'user', 'content': f'Question: {question}\n\nAnswer:\n{result.answer}\n\nCritique:\n{critique}'},
        ],
        temperature=0.2,
    )
    tracker.add(revise)
    final = revise.text
print('\nFINAL ANSWER:\n', final)

**Step 5 — Cost meter.** Every model call across the whole run, with a USD total.

In [ ]:
print(tracker.table())
print('\nTOTAL:', tracker.summary())

## What you built

A single agent that **plans its own retrieval**, **uses tools in a loop**, **critiques and revises** its answer, and **tracks cost** — all on top of the same provider-agnostic core from Phases 1–2, with no framework required.

**Phase 3 complete.** You now understand agents from the metal up: the loop, tools, ReAct, planning, reasoning search, reflection/Reflexion, self-correction, control, memory, safety, agentic RAG, and light multi-agent coordination — plus how these map onto real frameworks.

**Next phases (deferred):** Phase 4 — MCP · Phase 5 — Orchestration · Phase 6 — Memory · Phase 7 — Evaluation · Phase 8 — Production AI · Phase 9 — Architecture · Phase 10 — Real Systems.